# PROGRAMMING IN JAVA — ASSIGNMENT 2 (CO2)
## Google Colab Practical Notebook

**Name:** Archana V  
**Register Number:** 192521420  
**Faculty:** Ravi Pujitha  
**Academic Year:** 2026–2027  
**Course:** Programming in Java  
**Assignment:** Assignment 2 — CO2  
**Total Marks:** 50

---

### Coverage
Strings and StringBuilder; regular expressions; Collection Framework (`List`, `Queue`, `Deque`, `PriorityQueue`, `Set`, `Map`); `hashCode()`–`equals()` contract; Generics; `Comparable` and `Comparator`; and the `Collections` utility class.

### Important note about the supplied assignment PDF
The PDF contains **Q1–Q7**, not twelve separately numbered questions. To preserve every requirement without inventing questions, this notebook is organised into **12 work sections**: Q1–Q6, followed by Q7 parts A–F, in the exact order in the PDF. All Q1–Q7 requirements are covered.

### Running Java in Google Colab
Colab notebooks run Python by default. The first setup cell installs/uses a Java JDK; each Java source cell writes a `.java` file, and its following cell compiles and runs it. Run cells from top to bottom. The programs use standard Java syntax and are suitable for Java 17 or later.

### Evidence integrity
The assignment requires repeated averaged measurements and raw experimental evidence. Timing results depend on the machine and runtime. Therefore, benchmark cells calculate results when executed; **no timing or memory figures are pre-filled or claimed as measured**. Record the outputs from your own Colab session. Memory deltas from `Runtime` are approximate, not precise object-allocation measurements.

## Environment setup

The assignment asks students to execute every task. This cell checks Java availability. If Java is unavailable in a fresh Colab runtime, install the OpenJDK package first.

In [1]:
# Check Java runtime in Google Colab
!java -version
!javac -version

[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.016s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
javac 21.0.12.1


# TASK 1 — Q1: Building Text Under Load: String, StringBuilder or StringBuffer? (6 marks)

## 1.1 Prediction before implementation

- `String` is immutable. Repeated `result = result + field` creates new String values and copies previously accumulated characters. For a ticket with *n* similarly sized fields, the total copied characters can grow quadratically: **O(n²)**.
- `StringBuilder` is mutable and appends into a growable character buffer. Building a ticket is **O(n)** in the total number of characters (amortised append), with **O(n)** output storage.
- `StringBuffer` is also mutable, but its methods are synchronized. It is appropriate when the *same buffer* must be safely accessed by multiple threads, but synchronization adds overhead.
- A per-ticket builder is local to one request/thread; it does not need to be shared. A shared booking log needs synchronization or a thread-safe logging design. `StringBuffer` can protect individual append operations, but a complete log record should be appended atomically.

## 1.2 Three deliberately contrasting workloads

1. Short ticket: 6 fields, 1 ticket.
2. Heavy workload: 50 fields, 10,000 tickets.
3. Additional workload: 100 fields, 2,000 tickets.

The code measures elapsed time over repeated rounds using `System.nanoTime()`. It estimates intermediate String creation for repeated concatenation as one new accumulated result per append; this is an estimate, not JVM allocation instrumentation.

## 1.3 Pseudocode

```text
FOR each workload:
    FOR each implementation in String, StringBuilder, StringBuffer:
        repeat several rounds:
            start timer
            build every ticket by appending all fields
            stop timer
        report average elapsed time
```

In [2]:
%%writefile Task01.java
import java.util.*;

public class Task01 {
    static String[] fields(int count) {
        String[] a = new String[count];
        for (int i = 0; i < count; i++) a[i] = "FIELD" + i + "-VALUE";
        return a;
    }
    static String usingString(String[] fields) {
        String result = "";
        for (String field : fields) result = result + "|" + field;
        return result;
    }
    static String usingBuilder(String[] fields) {
        StringBuilder result = new StringBuilder();
        for (String field : fields) result.append('|').append(field);
        return result.toString();
    }
    static String usingBuffer(String[] fields) {
        StringBuffer result = new StringBuffer();
        for (String field : fields) result.append('|').append(field);
        return result.toString();
    }
    interface Builder { String build(String[] f); }
    static void benchmark(String label, int tickets, int fieldCount, Builder builder) {
        String[] f = fields(fieldCount);
        int rounds = 5;
        long total = 0;
        long beforeMemory = usedMemory();
        int checksum = 0;
        for (int r = 0; r < rounds; r++) {
            long start = System.nanoTime();
            for (int t = 0; t < tickets; t++) checksum += builder.build(f).length();
            total += System.nanoTime() - start;
        }
        long afterMemory = usedMemory();
        System.out.printf("%s | tickets=%d fields=%d | avg ms=%.3f | approx memory delta=%d bytes | checksum=%d%n",
            label, tickets, fieldCount, total / 5_000_000.0, afterMemory - beforeMemory, checksum);
    }
    static long usedMemory() {
        Runtime r = Runtime.getRuntime();
        return r.totalMemory() - r.freeMemory();
    }
    public static void main(String[] args) {
        int[][] workloads = {{1, 6}, {10000, 50}, {2000, 100}};
        for (int[] w : workloads) {
            System.out.println("\\nWORKLOAD: " + w[0] + " tickets, " + w[1] + " fields");
            benchmark("String +", w[0], w[1], Task01::usingString);
            benchmark("StringBuilder", w[0], w[1], Task01::usingBuilder);
            benchmark("StringBuffer", w[0], w[1], Task01::usingBuffer);
        }
        String[] sample = {"PNR123", "Archana", "S2", "B4", "450", "Chennai-Madurai"};
        System.out.println("\\nSample ticket: " + usingBuilder(sample));
    }
}

Writing Task01.java


In [3]:
!javac Task01.java
!java Task01

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
\nWORKLOAD: 1 tickets, 6 fields
String + | tickets=1 fields=6 | avg ms=2.361 | approx memory delta=209736 bytes | checksum=390
StringBuilder | tickets=1 fields=6 | avg ms=0.019 | approx memory delta=0 bytes | checksum=390
StringBuffer | tickets=1 fields=6 | avg ms=0.035 | approx memory delta=0 bytes | checksum=390
\nWORKLOAD: 10000 tickets,

## 1.4 Alternative concept: `String.join()`, `StringJoiner`, and `String.format()`

Example using `String.join()`:

```java
String ticket = String.join(" | ", pnr, passenger, coach, berth, fare, journey);
```

For a variable number of fields, use `StringJoiner`:

```java
StringJoiner joiner = new StringJoiner(" | ");
for (String field : fields) joiner.add(field);
String ticket = joiner.toString();
```

`String.join()` / `StringJoiner` avoid repeatedly rebuilding the accumulated prefix. Their work is linear in the total input/output characters. `String.format()` is convenient for fixed layouts but formatting has additional parsing/formatting overhead and is not the best default for a high-volume append loop.

| Approach | Time for n fields (similar field length) | Extra space |
|---|---:|---:|
| Repeated `+` in loop | O(n²) character copying | O(n²) cumulative temporary character creation; final result O(n) |
| `StringBuilder` | O(n) total characters, amortised | O(n) buffer + result |
| `StringBuffer` | O(n) total characters, synchronization overhead | O(n) buffer + result |
| `StringJoiner` / `String.join` | O(n) total characters | O(n) result/buffer |

**Design decision:** use a local `StringBuilder` for each ticket. For a shared log, use a dedicated thread-safe logging mechanism or synchronize the whole record append; `StringBuffer` is a simple synchronized option, but does not replace a robust logging subsystem.

**Record your evidence after running:** compare the three workload outputs. Do not conclude based on one run alone; the code averages five rounds, but repeat the entire cell if you need additional independent trials.

# TASK 2 — Q2: When Is ArrayList an Appropriate Engineering Choice? (4 marks)

## 2.1 Experiment design and prediction

Test three playlist sizes: 1,000, 10,000 and 50,000 songs. Compare:
- add at end;
- insert/remove at front;
- insert/remove in middle;
- access the k-th element;
- full iteration.

**Predictions:**

| Operation | ArrayList | LinkedList | ArrayDeque |
|---|---:|---:|---:|
| Add at end | O(1) amortised | O(1) | O(1) amortised |
| Insert at front | O(n) shifting | O(1) at ends | O(1) amortised |
| Insert middle by index | O(n) shifting | O(n) traversal + O(1) link | Not an indexed-list operation |
| Get k-th | O(1) | O(n) | No indexed access |
| Iterate all | O(n), cache-friendly | O(n) | O(n) |
| Remove front | O(n) shifting | O(1) | O(1) |

A linked node stores element and link reference(s), so it uses more per-element memory than an ArrayList array slot. ArrayDeque is designed for efficient operations at both ends and is a strong candidate for the “Up Next” queue.

## 2.2 Java experiment

The benchmark uses the same initial data size and reports timings. Timings are machine-dependent and should be interpreted as evidence, not universal constants.

In [4]:
%%writefile Task02.java
import java.util.*;

public class Task02 {
    static volatile long sink;
    static long time(Runnable action) {
        long s = System.nanoTime(); action.run(); return System.nanoTime() - s;
    }
    static void runSize(int n) {
        System.out.println("\\nSIZE = " + n);
        ArrayList<Integer> a = new ArrayList<>();
        LinkedList<Integer> l = new LinkedList<>();
        ArrayDeque<Integer> d = new ArrayDeque<>();
        for (int i=0;i<n;i++) { a.add(i); l.add(i); d.addLast(i); }

        System.out.printf("ArrayList add-end: %.3f ms%n", time(() -> { for(int i=0;i<5000;i++) a.add(i); })/1e6);
        System.out.printf("LinkedList add-end: %.3f ms%n", time(() -> { for(int i=0;i<5000;i++) l.add(i); })/1e6);
        System.out.printf("ArrayDeque add-end: %.3f ms%n", time(() -> { for(int i=0;i<5000;i++) d.addLast(i); })/1e6);

        System.out.printf("ArrayList front insert 100: %.3f ms%n", time(() -> { for(int i=0;i<100;i++) a.add(0,i); })/1e6);
        System.out.printf("LinkedList front insert 100: %.3f ms%n", time(() -> { for(int i=0;i<100;i++) l.addFirst(i); })/1e6);
        System.out.printf("ArrayDeque front insert 100: %.3f ms%n", time(() -> { for(int i=0;i<100;i++) d.addFirst(i); })/1e6);

        int mid = a.size()/2;
        System.out.printf("ArrayList middle insert/remove 100: %.3f ms%n", time(() -> {
            for(int i=0;i<100;i++){ a.add(a.size()/2,i); a.remove(a.size()/2); }
        })/1e6);
        System.out.printf("LinkedList middle indexed insert/remove 100: %.3f ms%n", time(() -> {
            for(int i=0;i<100;i++){ int p=l.size()/2; l.add(p,i); l.remove(p); }
        })/1e6);

        System.out.printf("ArrayList get 1000: %.3f ms%n", time(() -> {
            long x=0; for(int i=0;i<1000;i++) x += a.get(i%a.size()); sink=x;
        })/1e6);
        System.out.printf("LinkedList get 1000: %.3f ms%n", time(() -> {
            long x=0; for(int i=0;i<1000;i++) x += l.get(i%l.size()); sink=x;
        })/1e6);
        System.out.printf("ArrayList full iteration: %.3f ms%n", time(() -> {
            long x=0; for(int v:a) x+=v; sink=x;
        })/1e6);
        System.out.printf("LinkedList full iteration: %.3f ms%n", time(() -> {
            long x=0; for(int v:l) x+=v; sink=x;
        })/1e6);
    }
    public static void main(String[] args) {
        runSize(1000); runSize(10000); runSize(50000);
        ArrayDeque<String> upNext = new ArrayDeque<>();
        upNext.addFirst("Song B"); upNext.addLast("Song C"); upNext.addFirst("Song A");
        System.out.println("\\nUp Next (front to back): " + upNext);
        System.out.println("Play next: " + upNext.removeFirst());
    }
}

Writing Task02.java


In [5]:
!javac Task02.java
!java Task02

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
\nSIZE = 1000
ArrayList add-end: 4.550 ms
LinkedList add-end: 0.585 ms
ArrayDeque add-end: 0.737 ms
ArrayList front insert 100: 2.589 ms
LinkedList front insert 100: 0.051 ms
ArrayDeque front insert 100: 0.032 ms
ArrayList middle insert/remove 100: 3.104 ms
LinkedList middle indexed insert/remove 100: 22.162 ms
ArrayList get 1000: 0.398 ms


## 2.3 Engineering recommendation

`ArrayList` is reasonable when the playlist is read/accessed frequently by index, iterated often, and insertions/removals are mostly at the end. Stop recommending it when frequent front or middle edits dominate and shifting costs become a bottleneck. For “Up Next”, where items are added at both ends and removed from the front, use `ArrayDeque`.

**Plain array alternative:** maintain a logical size and manually shift elements right for insertion or left for removal. This has O(n) front/middle insertion/removal, O(1) indexed access, and fixed-capacity or resizing concerns. It offers low overhead but duplicates functionality already provided by collections.

# TASK 3 — Q3: Engineer the hashCode–equals Contract (6 marks)

## 3.1 Contract and test predictions

For objects considered equal by `equals()`, their `hashCode()` values **must** be equal. Unequal objects may share a hash code, but equal objects must never have different hash codes.

`HashSet` uses the hash code to choose a bucket and then uses `equals()` to distinguish matching objects in that bucket. If `equals()` is overridden but `hashCode()` is not, two logically equal Student objects can be placed in different buckets and both remain in the set.

Tests:
1. **Appears correct:** insert the same object reference twice. The set size should be 1 because identity is immediately equal.
2. **Expose duplicate:** insert two different Student objects with the same registration number. With the faulty implementation, their identity-based hash codes can differ; duplicates may appear. This is a deliberate defect demonstration, and exact faulty output is not guaranteed because hash codes are runtime-dependent.
3. **Mutation test:** demonstrate why fields used in equality/hash must not change while an object is stored in a hash set. The corrected class below makes identity fields `final`.

## 3.2 Complete the Code — corrected implementation

In [6]:
%%writefile Task03.java
import java.util.*;

class Student {
    private final String regNo;
    private final String name;
    Student(String regNo, String name) { this.regNo = regNo; this.name = name; }

    @Override public boolean equals(Object o) {
        if (this == o) return true;
        if (!(o instanceof Student)) return false;
        Student other = (Student) o;
        return Objects.equals(regNo, other.regNo);
    }
    @Override public int hashCode() { return Objects.hash(regNo); }
    @Override public String toString() { return regNo + " " + name; }
}
public class Task03 {
    public static void main(String[] args) {
        Set<Student> present = new HashSet<>();
        present.add(new Student("192311001", "Arun"));
        present.add(new Student("192311002", "Divya"));
        present.add(new Student("192311001", "Arun"));
        System.out.println("Unique students: " + present.size());

        Student same = new Student("192311003", "Kavi");
        Set<Student> identityTest = new HashSet<>();
        identityTest.add(same); identityTest.add(same);
        System.out.println("Same reference inserted twice: " + identityTest.size());

        Set<Student> verification = new HashSet<>();
        verification.add(new Student("192311004", "Mira"));
        verification.add(new Student("192311004", "Mira"));
        System.out.println("Two equal objects after correction: " + verification.size());
    }
}

Writing Task03.java


In [7]:
!javac Task03.java
!java Task03

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Unique students: 2
Same reference inserted twice: 1
Two equal objects after correction: 1


## 3.3 Before/after TODO evidence

The supplied PDF's `Attendance` example expects:

```text
Unique students: 2
```

Before TODO 1, `hashCode()` is inherited from `Object`. Two separate Student instances that have the same registration number may have different hash codes, so `HashSet` may retain both. The exact faulty result should be recorded from an intentionally faulty run; it is not deterministic and must not be invented.

After TODO 1, use:

```java
@Override public int hashCode() {
    return Objects.hash(regNo);
}
```

The corrected result is `Unique students: 2`.

## 3.4 Alternative concepts

- `TreeSet<Student>` with a comparator on `regNo`: de-duplicates when comparator returns zero; add/search/remove are O(log n). Ordering is sorted by registration number.
- `HashMap<String, Student>` keyed by registration number: `putIfAbsent(regNo, student)` prevents duplicate keys; average O(1) insertion/lookup.
- `HashSet`: average O(1) add/contains; O(n) storage for n unique students.
- `TreeSet`: O(log n) operations; O(n) storage plus tree-node links.
- `HashMap`: average O(1) operations; O(n) storage for keys, values and table structure.

Never mutate a field participating in `equals()` or `hashCode()` while the object is in a hash-based collection. If it changes, the object may remain in the bucket associated with its old hash and become difficult to find/remove.

# TASK 4 — Q4: Choosing a Map for a Pharmacy Inventory (6 marks)

## 4.1 Design reasoning

- `HashMap`: average O(1) `put`/`get`; iteration O(n), with no guaranteed order.
- `TreeMap`: O(log n) `put`/`get`; iteration O(n) in sorted key order.
- `LinkedHashMap`: average O(1) `put`/`get`; iteration O(n) in insertion order.
- Code-order report: `TreeMap` naturally satisfies it.
- Added-order report: insertion-order `LinkedHashMap` naturally satisfies it.
- If the wrong map is used, copy keys/entries and sort them (typically O(n log n)) or maintain a second index.

The benchmark below uses 3 inventory sizes and performs 100,000 lookups per size. It times each report generation and checks the order.

In [8]:
%%writefile Task04.java
import java.util.*;

public class Task04 {
    static volatile int sink;
    static String code(int i) { return String.format("MED%06d", i); }
    static void test(int n) {
        Map<String,Integer> hash = new HashMap<>();
        Map<String,Integer> tree = new TreeMap<>();
        Map<String,Integer> linked = new LinkedHashMap<>();
        List<String> keys = new ArrayList<>();
        for (int i=n-1;i>=0;i--) keys.add(code(i)); // deliberately reverse insertion order
        for (String k:keys) { hash.put(k,100); tree.put(k,100); linked.put(k,100); }

        Random random = new Random(42);
        List<String> queries = new ArrayList<>();
        for(int i=0;i<100000;i++) queries.add(code(random.nextInt(n)));
        long s=System.nanoTime(); int sum=0;
        for(String q:queries) sum += hash.get(q);
        long hashLookup=System.nanoTime()-s; sink=sum;
        s=System.nanoTime(); sum=0;
        for(String q:queries) sum += tree.get(q);
        long treeLookup=System.nanoTime()-s; sink=sum;
        s=System.nanoTime(); sum=0;
        for(String q:queries) sum += linked.get(q);
        long linkedLookup=System.nanoTime()-s; sink=sum;

        s=System.nanoTime(); List<String> sorted=new ArrayList<>(hash.keySet()); Collections.sort(sorted);
        long sortedReport=System.nanoTime()-s;
        s=System.nanoTime(); List<String> treeReport=new ArrayList<>(tree.keySet());
        long treeReportTime=System.nanoTime()-s;
        s=System.nanoTime(); List<String> insertion=new ArrayList<>(linked.keySet());
        long insertionReport=System.nanoTime()-s;
        System.out.printf("\\nMedicines=%d | lookup 100k ms: HashMap %.3f, TreeMap %.3f, LinkedHashMap %.3f%n",
            n,hashLookup/1e6,treeLookup/1e6,linkedLookup/1e6);
        System.out.printf("Code-order report: TreeMap %.3f ms; sort HashMap keys %.3f ms; correct=%b%n",
            treeReportTime/1e6,sortedReport/1e6,treeReport.equals(sorted));
        System.out.printf("Insertion-order report: LinkedHashMap %.3f ms; correct=%b%n",
            insertionReport/1e6,insertion.equals(keys));
        System.out.println("First code in sorted report: " + tree.keySet().iterator().next());
        System.out.println("First code in insertion report: " + insertion.get(0));
    }
    public static void main(String[] args) {
        test(1000); test(10000); test(50000);
    }
}

Writing Task04.java


In [9]:
!javac Task04.java
!java Task04

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
\nMedicines=1000 | lookup 100k ms: HashMap 45.875, TreeMap 90.695, LinkedHashMap 28.158
Code-order report: TreeMap 0.693 ms; sort HashMap keys 3.518 ms; correct=true
Insertion-order report: LinkedHashMap 0.355 ms; correct=true
First code in sorted report: MED000000
First code in insertion report: MED000999
\nMedicines=10000 | lookup 100k ms

## 4.2 Deployment decision and scalability

Define the acceptable single-lookup target with the actual pharmacy requirement before choosing a map. For example, a project may set a target of less than 10 ms per lookup, but this is a **design target**, not a measured fact from this notebook. The remaining costs include hashing/comparison, memory access, resizing, report traversal and I/O.

For the two required reports, a practical design is:
- `TreeMap` for the code-sorted report;
- `LinkedHashMap` for insertion-order report;
- or maintain both indexes if both report orders are frequent and the extra memory/update work is acceptable.

`Hashtable` is generally not preferred in new code because it is a legacy synchronized collection; modern alternatives provide clearer concurrency choices (`ConcurrentHashMap`) or ordinary unsynchronized maps where external coordination is sufficient.

## 4.3 Alternative: sorted array + `Arrays.binarySearch()`

- Lookup: O(log n).
- Insertion: O(n) because elements may need shifting to preserve sorted order.
- Code-order report: O(n) traversal (already sorted).
- Space: O(n) array slots; resizing may temporarily require another array.
- `HashMap`: average O(1) lookup, no sorted order.
- `TreeMap`: O(log n) lookup/insertion and sorted iteration, with tree-node overhead.

Example:

```java
int index = Arrays.binarySearch(codes, targetCode);
if (index >= 0) System.out.println(stock[index]);
```

# TASK 5 — Q5: Derive a Generic Ranking Component from a Principle (7 marks)

## 5.1 Design

Ranking rule:
1. Points descending.
2. Goal difference descending.
3. Team name ascending (A–Z).

A separate `Comparator<Team>` is preferable here because the ranking policy may change for different competitions and the same component is intended for teams, players, colleges and quiz teams. `Comparable<Team>` is appropriate when one natural ordering is intrinsic and stable.

`Leaderboard<T>` accepts `Comparator<? super T>` so a comparator for a supertype can compare T values. `List<? extends T>` is useful when accepting a source list of T or subtype objects for reading/copying.

## 5.2 Complete the Code

In [10]:
%%writefile Task05.java
import java.util.*;

class Team {
    final String name; final int points; final int goalDiff;
    Team(String name, int points, int goalDiff) {
        this.name=name; this.points=points; this.goalDiff=goalDiff;
    }
    @Override public String toString() { return name+"("+points+","+goalDiff+")"; }
}
class Leaderboard<T> {
    private final List<T> items = new ArrayList<>();
    private final Comparator<? super T> order;
    Leaderboard(Comparator<? super T> order) { this.order=order; }
    void add(T item) { items.add(item); }
    List<T> ranking() {
        List<T> copy = new ArrayList<>(items);
        copy.sort(order);
        return copy;
    }
    void addAll(List<? extends T> incoming) { items.addAll(incoming); }
}
public class Task05 {
    public static void main(String[] args) {
        Comparator<Team> byRank =
            Comparator.comparingInt((Team t)->t.points).reversed()
            .thenComparing(Comparator.comparingInt((Team t)->t.goalDiff).reversed())
            .thenComparing(t->t.name);

        Leaderboard<Team> board = new Leaderboard<>(byRank);
        board.add(new Team("Panthers",9,4));
        board.add(new Team("Eagles",9,6));
        board.add(new Team("Cobras",6,1));
        board.add(new Team("Bisons",9,4));
        System.out.println(board.ranking());

        List<Team> more = Arrays.asList(new Team("Falcons",9,4), new Team("Tigers",12,0));
        board.addAll(more);
        System.out.println("After adding more teams: " + board.ranking());
    }
}

Writing Task05.java


In [11]:
!javac Task05.java
!java Task05

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[Eagles(9,6), Bisons(9,4), Panthers(9,4), Cobras(6,1)]
After adding more teams: [Tigers(12,0), Eagles(9,6), Bisons(9,4), Falcons(9,4), Panthers(9,4), Cobras(6,1)]


## 5.3 Correctness and deliberate faulty-comparator test

The comparator compares points first, so a team with more points ranks first. If points tie, it compares goal difference descending. If both tie, it compares names alphabetically. The chain is lexicographic and consistent: each later criterion is consulted only when all earlier criteria compare equal.

The supplied expected output is:

```text
[Eagles(9,6), Bisons(9,4), Panthers(9,4), Cobras(6,1)]
```

Bisons is above Panthers because their points and goal difference are equal, and `"Bisons"` comes before `"Panthers"` alphabetically.

A faulty comparator such as `return b.points - a.points;` can overflow for extreme integer values. Use `Integer.compare(b.points, a.points)` or `Comparator.comparingInt(...).reversed()` instead. A comparator that compares only points ignores tie-breakers and may treat distinct teams as equal in sorted sets.

## 5.4 Alternative concepts and complexity

- `Comparator` chain: one sort, O(n log n) comparisons; sorting a copied list uses O(n) extra space (implementation-dependent details).
- `Comparable<Team>`: put the natural rule in `compareTo`; same O(n log n) sort complexity, but less flexible if multiple ranking policies exist.
- Three stable sorts: sort by name, then goal difference descending, then points descending. Each sort is O(n log n), so three sorts remain O(n log n) asymptotically but do more work and require stable sorting to preserve earlier tie-breaks.
- A generic `Leaderboard<T>` with an injected comparator is easiest to reuse for players and colleges because the ranking policy is supplied from outside the generic class.

# TASK 6 — Q6: Priority, Order and Fairness in a Queue (8 marks)

## 6.1 Design

A plain FIFO queue serves the earliest arrival regardless of severity, so it cannot guarantee that the most severe patient is treated next. Use `PriorityQueue` with:
- severity ascending (1 is most severe);
- arrival counter ascending for equal severity.

The arrival counter provides stable FIFO ordering within the same severity. A `PriorityQueue` iterator or direct `toString()` is **not** guaranteed to display sorted treatment order; repeatedly call `poll()` on a copy to display the order.

## 6.2 Java implementation — about 8 patients, plus benchmark

In [12]:
%%writefile Task06.java
import java.util.*;

class Patient {
    final String name; final int severity; final long arrival;
    Patient(String name,int severity,long arrival) {
        this.name=name; this.severity=severity; this.arrival=arrival;
    }
    @Override public String toString(){ return name+"(S"+severity+")"; }
}
public class Task06 {
    private long counter=0;
    private final PriorityQueue<Patient> queue = new PriorityQueue<>(
        Comparator.comparingInt((Patient p)->p.severity)
                  .thenComparingLong(p->p.arrival));
    void admit(String name,int severity){ queue.add(new Patient(name,severity,counter++)); }
    Patient next(){ return queue.poll(); }
    List<Patient> waitingInTreatmentOrder(){
        PriorityQueue<Patient> copy=new PriorityQueue<>(queue);
        List<Patient> result=new ArrayList<>();
        while(!copy.isEmpty()) result.add(copy.poll());
        return result;
    }
    static long benchmark(int n) {
        PriorityQueue<Integer> q=new PriorityQueue<>();
        long start=System.nanoTime();
        for(int i=0;i<n;i++) q.add(n-i);
        while(!q.isEmpty()) q.poll();
        return System.nanoTime()-start;
    }
    public static void main(String[] args) {
        Task06 er=new Task06();
        er.admit("Ravi",3); er.admit("Meena",1); er.admit("John",3);
        er.admit("Asha",5); er.admit("Kumar",1); er.admit("Lata",2);
        er.admit("Nila",4); er.admit("Dev",2);
        System.out.println("Treatment order: "+er.waitingInTreatmentOrder());
        StringBuilder order=new StringBuilder();
        Patient p;
        while((p=er.next())!=null) order.append(p).append(' ');
        System.out.println("Served: "+order.toString().trim());
        for(int n:new int[]{1000,10000,50000})
            System.out.printf("Add and remove %d patients: %.3f ms%n",n,benchmark(n)/1e6);
    }
}

Writing Task06.java


In [13]:
!javac Task06.java
!java Task06

[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Treatment order: [Meena(S1), Kumar(S1), Lata(S2), Dev(S2), Ravi(S3), John(S3), Nila(S4), Asha(S5)]
Served: Meena(S1) Kumar(S1) Lata(S2) Dev(S2) Ravi(S3) John(S3) Nila(S4) Asha(S5)
Add and remove 1000 patients: 9.066 ms
Add and remove 10000 patients: 29.597 ms
Add and remove 50000 patients: 58.085 ms


## 6.3 Expected order for the six patients in the supplied PDF

```text
Meena(S1) Kumar(S1) Lata(S2) Ravi(S3) John(S3) Asha(S5)
```

Meena precedes Kumar because both have severity 1 and Meena arrived first. Ravi precedes John for the same reason at severity 3.

## 6.4 Growth, starvation and fairness

- `PriorityQueue.add()`: O(log n).
- `PriorityQueue.poll()`: O(log n).
- Peek next: O(1).
- Storage: O(n).

A severity-5 patient can starve if higher-severity patients continually arrive. A fair policy can increase a patient's effective priority after waiting beyond a threshold (ageing), or reserve service capacity for lower-severity categories while maintaining emergency safeguards. The policy must be clinically approved; the data structure alone cannot decide medical fairness.

A city hospital receiving 5,000 patients/day is not automatically too large for a priority queue: O(log n) insertion/removal is scalable at that volume. Real deployment also requires concurrency control, audit logs, persistent storage, clinical rules, emergency overrides and monitoring.

## 6.5 Alternative concepts

| Design | Admit | Serve next | Space |
|---|---:|---:|---:|
| PriorityQueue | O(log n) | O(log n) | O(n) |
| Sorted LinkedList | O(n) search/insertion | O(1) remove first | O(n), node overhead |
| Five FIFO queues (one per severity) | O(1) | O(1) to inspect severity queues (fixed 5 levels) | O(n) |

Five FIFO queues work well when severity has a small fixed number of levels. Select the first non-empty queue from severity 1 through 5. For more complex dynamic priorities, a priority queue is more flexible.

# TASK 7 — Q7: Diagnose, Redesign and Validate (13 marks)

The PDF asks the student to select **one** solution from Q1–Q6 that has a significant limitation as scale or usage changes. This notebook selects Q2's playlist design: repeated front insertions in an `ArrayList` cause shifting of existing elements.

## A. Diagnose the Bottleneck (2 marks)

**Selected original solution:** `ArrayList` used as an “Up Next” queue with frequent insertion/removal at the front.

**Diagnosis:** each front insertion shifts the existing elements one position to the right; each front removal shifts remaining elements left. With n elements, one such operation is O(n). Repeating many front operations causes repeated shifting and unnecessary data movement. The limitation is caused by the contiguous array representation and index-based layout.

**Evidence to record:** use the Task 2 benchmark output at 1,000, 10,000 and 50,000 elements. Compare front insertion timings. Do not copy a timing from another machine.

## B. Define the Design Objective (2 marks)

Redesign the “Up Next” queue to support adding at both ends and removing the next song from the front without shifting the entire collection.

**Measurable objective:** for large queues, reduce front add/remove operation complexity from O(n) to O(1) amortised, while preserving the order of songs.

## C. Design the Revised Solution (4 marks)

Use `ArrayDeque<String>`:
- `addFirst()` / `addLast()` add at either end;
- `removeFirst()` serves the next song;
- `peekFirst()` inspects the next song.

**Pseudocode**

```text
Create an empty deque
For each incoming song:
    if song is urgent: add it at the front
    otherwise: add it at the back
While the deque is not empty:
    play and remove the first song
```

**Conceptual class diagram**

```text
          UpNextQueue
        ----------------
        - songs: Deque<Song>
        ----------------
        + addUrgent(Song)
        + addSong(Song)
        + playNext(): Song
        + peekNext(): Song
        + isEmpty(): boolean
                 |
                 | uses
                 v
             ArrayDeque
```

**Trade-offs:** ArrayDeque has no indexed access and does not permit null elements. It may resize its backing array, and it is not thread-safe. If multiple threads modify it, external synchronization or a concurrent queue design is needed.

## D. Predict Before Implementing (1 mark)

Prediction: for front insertion/removal, ArrayDeque should avoid per-operation shifting and show O(1) amortised end operations, while ArrayList front operations are O(n). Full traversal remains O(n). This prediction will be tested with identical operation counts and queue sizes.

## E. Implement and Validate the Redesign (2 marks)

The following program compares repeated front insertions and removals using `ArrayList` and `ArrayDeque` under the same test sizes. It also verifies that both structures produce the same logical result. Run it in Colab and preserve the raw output as your experimental evidence.

In [14]:
%%writefile Task07.java
import java.util.*;

public class Task07 {
    static volatile int sink;
    static long testArrayList(int n) {
        List<Integer> list=new ArrayList<>();
        long start=System.nanoTime();
        for(int i=0;i<n;i++) list.add(0,i);
        while(!list.isEmpty()) list.remove(0);
        sink=list.size();
        return System.nanoTime()-start;
    }
    static long testArrayDeque(int n) {
        Deque<Integer> deque=new ArrayDeque<>();
        long start=System.nanoTime();
        for(int i=0;i<n;i++) deque.addFirst(i);
        while(!deque.isEmpty()) deque.removeFirst();
        sink=deque.size();
        return System.nanoTime()-start;
    }
    public static void main(String[] args) {
        for(int n:new int[]{1000,10000,50000}) {
            long a=testArrayList(n);
            long d=testArrayDeque(n);
            System.out.printf("n=%d | ArrayList front operations=%.3f ms | ArrayDeque=%.3f ms%n",
                n,a/1e6,d/1e6);
        }
        List<String> original=new ArrayList<>();
        Deque<String> redesigned=new ArrayDeque<>();
        original.add(0,"Song B"); original.add("Song C");
        original.add(0,"Song A");
        redesigned.addFirst("Song B"); redesigned.addLast("Song C");
        redesigned.addFirst("Song A");
        System.out.println("Original order:   "+original);
        System.out.println("Redesigned order: "+redesigned);
        System.out.println("Same order: "+original.equals(new ArrayList<>(redesigned)));
    }
}

Writing Task07.java


In [15]:
!javac Task07.java
!java Task07

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.000s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
n=1000 | ArrayList front operations=1.786 ms | ArrayDeque=0.891 ms
n=10000 | ArrayList front operations=12.586 ms | ArrayDeque=3.138 ms
n=50000 | ArrayList front operations=249.714 ms | ArrayDeque=7.584 ms
Original order:   [Song A, Song B, Song C]
Redesigned order: [Song A, Song B, Song C]
Same order: true


## F. Defend the Final Design (2 marks)

Complete this evaluation after running Task 7:

1. **Was the prediction supported?** Compare the measured timings for all three sizes. State whether ArrayDeque's end operations grew more slowly in your run.
2. **Magnitude of improvement:** Calculate `ArrayList time / ArrayDeque time` for each size from your measured outputs. Do not report a ratio unless you calculate it from your own run.
3. **New costs/trade-offs:** ArrayDeque does not support indexed access, disallows null, can resize, and is not thread-safe.
4. **When the original is preferable:** ArrayList remains preferable when random indexed access is important and front/middle insertion/removal is rare.

A redesign need not always be faster in every small test. JVM warm-up, garbage collection, machine load and small input sizes can affect timing. Explain any result that does not match the prediction rather than changing the data.

### Evidence record (fill after execution)

| Queue size | ArrayList measured time | ArrayDeque measured time | Ratio (ArrayList / Deque) |
|---:|---:|---:|---:|
| 1,000 | Record output | Record output | Calculate |
| 10,000 | Record output | Record output | Calculate |
| 50,000 | Record output | Record output | Calculate |

Conclusion:

This assignment demonstrates how Java strings, lists, sets, maps, generics, comparators and queues behave under different workloads. The main engineering lesson is to select a data structure based on the operations and ordering requirements, then validate the choice using repeated measurements and correctness tests. Final conclusions must be supported by the student's own execution evidence.


